In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

# 1. Load dataset
df = pd.read_csv("amazon_prime_titles.csv")

print("Dataset Shape:", df.shape)
print(df.head())
print("\nData Types:\n", df.dtypes)
print("\nMissing Values:\n", df.isnull().sum())

# 2. Remove duplicate records
print("\nDuplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().copy()

# 3. Clean column names and text
df.columns = df.columns.str.strip()

for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].astype("string").str.strip()
    df[col] = df[col].replace("", pd.NA)

# 4. Handle missing values
# Fill categorical columns with Unknown
for col in ["type", "country", "rating", "director",
            "cast", "listed_in", "description"]:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

# Convert data types
df["release_year"] = pd.to_numeric(
    df["release_year"], errors="coerce"
)

df["date_added"] = pd.to_datetime(
    df["date_added"], errors="coerce"
)

# Extract numeric duration
df["duration_value"] = pd.to_numeric(
    df["duration"].astype("string").str.extract(r"(\d+)")[0],
    errors="coerce"
)

print("\nMissing Values After Cleaning:\n", df.isnull().sum())
print("\nCleaned Dataset Shape:", df.shape)

# 5. Detect outliers using IQR method
def detect_outliers(data, column):
    values = pd.to_numeric(data[column], errors="coerce").dropna()

    if values.empty:
        print(f"{column}: No numeric values available")
        return

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = values[(values < lower) | (values > upper)]

    print(f"\n{column} outliers: {len(outliers)}")
    print("Lower bound:", lower)
    print("Upper bound:", upper)

detect_outliers(df, "release_year")

# Check movie durations separately
movies = df[df["type"] == "Movie"]
detect_outliers(movies, "duration_value")

# Set graph size
plt.rcParams["figure.figsize"] = (10, 6)

# GRAPH 1: Movies vs TV Shows
df["type"].value_counts().plot(kind="bar")
plt.title("Movies vs TV Shows")
plt.xlabel("Content Type")
plt.ylabel("Number of Titles")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# GRAPH 2: Titles by Release Year
df["release_year"].dropna().astype(int).value_counts().sort_index().plot()
plt.title("Titles by Release Year")
plt.xlabel("Release Year")
plt.ylabel("Number of Titles")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# GRAPH 3: Top 10 Countries
countries = (
    df.loc[df["country"] != "Unknown", "country"]
    .dropna().str.split(",").explode().str.strip()
)
countries.value_counts().head(10).sort_values().plot(kind="barh")
plt.title("Top 10 Production Countries")
plt.xlabel("Number of Country Listings")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

# GRAPH 4: Content Ratings
df["rating"].value_counts().head(12).sort_values().plot(kind="barh")
plt.title("Content Rating Distribution")
plt.xlabel("Number of Titles")
plt.ylabel("Rating")
plt.tight_layout()
plt.show()

# GRAPH 5: Top 10 Genres
genres = (
    df.loc[df["listed_in"] != "Unknown", "listed_in"]
    .dropna().str.split(",").explode().str.strip()
)
genres.value_counts().head(10).sort_values().plot(kind="barh")
plt.title("Top 10 Genres")
plt.xlabel("Number of Genre Listings")
plt.ylabel("Genre")
plt.tight_layout()
plt.show()

# GRAPH 6: Movie Duration Distribution
movie_duration = movies["duration_value"].dropna()
plt.hist(movie_duration, bins=30, edgecolor="black")
plt.title("Movie Duration Distribution")
plt.xlabel("Duration (Minutes)")
plt.ylabel("Number of Movies")
plt.tight_layout()
plt.show()

# GRAPH 7: TV Shows by Seasons
tv = df[df["type"] == "TV Show"]
seasons = tv["duration_value"].dropna().astype(int)
seasons.value_counts().sort_index().plot(kind="bar")
plt.title("TV Shows by Number of Seasons")
plt.xlabel("Number of Seasons")
plt.ylabel("Number of TV Shows")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# GRAPH 8: Titles Added by Year
df["date_added"].dropna().dt.year.value_counts().sort_index().plot(
    marker="o"
)
plt.title("Titles Added to Amazon Prime by Year")
plt.xlabel("Year Added")
plt.ylabel("Number of Titles")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# 6. Final report
print("\nFinal Dataset Shape:", df.shape)
print("\nRemaining Missing Values:\n", df.isnull().sum())
print("\nEDA Completed Successfully!")